## Embeddings: Funciones

Constantes y funciones compartidas

### Installs

- `gensim` (Word2Vec de los paseos) y `umap-learn` (proyección 2D) no vienen en la imagen del cluster

- Se fijan las versiones de la imagen (`numpy`, `scipy`, `scikit-learn`) para no alterar lo que corrieron las partes anteriores

In [ ]:
%pip install -q gensim==4.3.2 umap-learn==0.5.6 numpy==1.26.4 scipy==1.12.0 scikit-learn==1.4.1.post1

### Imports

In [ ]:
import json, zlib, tracemalloc
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from sklearn.utils.extmath import randomized_svd
from scipy.stats import spearmanr
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score, silhouette_score,
                             roc_auc_score, average_precision_score)

### Constantes

- `SEED` = 42: misma semilla en paseos, Word2Vec, UMAP, KMeans y muestras

- `datos_dir`: datos que pasan de un notebook al siguiente

- `clus_dir`: salidas de `02_Clustering` (etiquetas de KMeans, DBSCAN, HDBSCAN y Ward)

- `pr_dir`: salidas de `04_PageRank` (PageRank y HITS por nodo)

- `MAX_PART` y `MIN_COMP`: los mismos de `01_Trans/c_uso` para armar la co-participación por año

- `N_SIL` = 3,000: puntos para estimar la silueta

In [ ]:
SEED = 42
datos_dir = Path.cwd() / "temp" / "datos"
datos_dir.mkdir(parents=True, exist_ok=True)
clus_dir = proyecto / "02_Clustering" / "temp"
pr_dir = proyecto / "04_PageRank" / "temp" / "datos"
MAX_PART = 30
MIN_COMP = 2
N_SIL = 3000

### Acople con la Parte V (comunidades)

- La Parte V la desarrolla otro integrante; aquí **no se calcula ninguna comunidad**: se leen sus salidas de `05_Comunidad/temp/datos`

- `nodos.parquet` (clave de proveedor por `idx`) + `particion.parquet` (etiquetas por `idx`); la partición usada es la **principal** que fija su `config.json`

- Si la ruta no existe, el cruce se omite y los notebooks corren igual contra las Partes II y IV

In [ ]:
COMUNIDADES = {
    "ruta": proyecto / "05_Comunidad" / "temp" / "datos",   # nodos.parquet + particion.parquet + config.json
    "llave": "clave",                                        # misma llave canónica de proveedor
}

### Parámetros elegidos

| Parámetro | Valor | Por qué | Sección |
|---|---|---|---|
| `PCT_HUB` | 0.999 | Retira 61 postores masivos (11.4% de la masa de visita) sin romper el grafo | `a_grafo` |
| `DIM` | 32 | Mejor AUC sin fuga (2023 → 2024) que 64 y 128 | `b_node2vec` |
| `N_PASEOS`, `LARGO`, `VENTANA`, `EPOCAS` | 10, 40, 5, 1 | Artículo original con paseo más corto; costo lineal en el largo | `b_node2vec` |
| `RANGO_P`, `RANGO_Q` | 0.25, 1, 4 | Extremos de BFS y DFS; con 0.5-2 no se distinguían del ruido | `b_node2vec` |
| `VECINOS_UMAP` | 30 | Mayor confiabilidad (0.985) y más rápido que 50 | `c_proyeccion` |
| `K_LAT` | 6 | Mayor *k* con partición estable entre semillas (ARI 0.99) | `c_proyeccion` |
| `CONF_REF` | p = 1, q = 1 | Referencia neutra; las demás dan perfiles casi iguales | `c_proyeccion` |
| Protocolo de enlaces | embedding 2023, train 2024, test 2025 | Sin fuga: cada paso solo ve años anteriores | `d_enlaces` |
| `FRAC_DIFICIL` | 0.5 | Mitad de negativos a 2 saltos: evalúa el caso realista | `d_enlaces` |
| Modelo de enlaces | random forest, Hadamard | Mejor PR-AUC en test (0.72) | `d_enlaces` |

### Configuración entre notebooks

`config.json` guarda lo que cada notebook decide (subgrafo, grilla, dimensión…)

In [ ]:
def leer_config():
    ruta = datos_dir / "config.json"
    return json.loads(ruta.read_text()) if ruta.exists() else {}

def guardar_config(config):
    (datos_dir / "config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False))

### Grafo

- `matriz_adyacencia`: adyacencia N × N simétrica y ponderada (CSR con índices ordenados)

- `cargar_coparticipacion`: vértices con índice entero `idx` y aristas `i < j` del artefacto de `01_Trans`

- `componente_gigante`: índices de los nodos en la componente más grande

In [ ]:
def matriz_adyacencia(i, j, w, n):
    A = sp.coo_matrix((np.r_[w, w], (np.r_[i, j], np.r_[j, i])), shape=(n, n)).tocsr()
    A.sum_duplicates()
    A.sort_indices()
    return A

def cargar_coparticipacion():
    v = pd.read_parquet(artef_dir / "grafo_coparticipacion" / "vertices").sort_values("id", ignore_index=True)
    a = pd.read_parquet(artef_dir / "grafo_coparticipacion" / "aristas")
    v["idx"] = np.arange(len(v))
    v["clave"] = v.id.str[2:]
    pos = pd.Series(v.idx.to_numpy(), index=v.id)
    a = a.assign(i=a.src.map(pos), j=a.dst.map(pos))
    assert a[["i", "j"]].notna().all().all(), "hay aristas con extremos fuera de los vértices"
    return v, a.astype({"i": int, "j": int})

def componente_gigante(A):
    _, comp = connected_components(A, directed=False)
    return np.flatnonzero(comp == np.bincount(comp).argmax())

### Node2Vec

**Paseo de segundo orden:** estando en `v` y viniendo de `t`, el siguiente nodo `x` (vecino de `v`) se elige con peso w(v, x) · α(t, x)

| Caso | α | Efecto |
|---|---|---|
| x = t (volver) | 1/p | p chico → se queda cerca |
| x vecino de t | 1 | se mueve dentro del mismo grupo |
| x lejos de t | 1/q | q chico → se aleja (DFS); q grande → BFS |

- **Muestreo por rechazo:** se propone x con probabilidad ∝ w(v, x) y se acepta con α / máx(α); no hay que precalcular tablas de alias por arista

- Todos los paseos avanzan **a la vez** con numpy: un paso = una operación vectorizada sobre todos los caminantes

- **Vecindad de t:** búsqueda binaria de la llave `t·N + x` en las aristas ordenadas

In [ ]:
def _vecino_ponderado(A, cum, nodos, rng):
    ini, fin = A.indptr[nodos], A.indptr[nodos + 1]
    base = np.where(ini > 0, cum[ini - 1], 0.0)
    obj = base + rng.random(len(nodos)) * (cum[fin - 1] - base)
    k = np.searchsorted(cum, obj, side="right")
    return A.indices[np.clip(k, ini, fin - 1)]

def node2vec_paseos(A, p, q, n_paseos, largo, seed=SEED):
    rng = np.random.default_rng(seed)
    n = A.shape[0]
    cum = np.cumsum(A.data)
    llaves = np.repeat(np.arange(n, dtype=np.int64), np.diff(A.indptr)) * n + A.indices
    a_max = max(1 / p, 1.0, 1 / q)
    inicio = np.tile(np.arange(n), n_paseos)
    rng.shuffle(inicio)
    P = np.empty((len(inicio), largo), dtype=np.int32)
    P[:, 0] = inicio
    P[:, 1] = _vecino_ponderado(A, cum, inicio, rng)
    for s in range(2, largo):
        t, v = P[:, s - 2].astype(np.int64), P[:, s - 1]
        x = np.empty(len(v), dtype=np.int32)
        pend = np.arange(len(v))
        while len(pend):
            x_ = _vecino_ponderado(A, cum, v[pend], rng)
            k = t[pend] * n + x_
            pos = np.minimum(np.searchsorted(llaves, k), len(llaves) - 1)
            alpha = np.where(x_ == t[pend], 1 / p, np.where(llaves[pos] == k, 1.0, 1 / q))
            ok = rng.random(len(pend)) * a_max < alpha
            x[pend[ok]] = x_[ok]
            pend = pend[~ok]
        P[:, s] = x
    return P

### Word2Vec sobre los paseos

- **Skip-gram** con muestreo negativo: dos nodos quedan cerca si aparecen en la misma ventana de los paseos

- Los paseos van a un archivo de texto (`corpus_file`): Word2Vec los lee en C, sin millones de listas de Python

- `workers=1` y `hashfxn` determinista (CRC32): con la misma semilla, el mismo embedding en cada ejecución

In [ ]:
def _hash_estable(s):
    return zlib.crc32(s.encode())

def entrenar_w2v(P, dim, ventana, epocas, seed=SEED):
    corpus = datos_dir / "paseos.txt"
    np.savetxt(corpus, P, fmt="%d")
    m = Word2Vec(corpus_file=str(corpus), vector_size=dim, window=ventana, min_count=0, sg=1, negative=5,
                 epochs=epocas, workers=1, seed=seed, hashfxn=_hash_estable)
    corpus.unlink()
    n = int(P.max()) + 1
    E = np.zeros((n, dim), dtype=np.float32)
    E[[int(k) for k in m.wv.index_to_key]] = m.wv.vectors
    return E

def log(msg, t0=None):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def node2vec(A, p, q, dim, n_paseos, largo, ventana, epocas, seed=SEED):
    t0 = time.time()
    P = node2vec_paseos(A, p, q, n_paseos, largo, seed)
    t1 = time.time()
    E = entrenar_w2v(P, dim, ventana, epocas, seed)
    return E, {"t_paseos_s": t1 - t0, "t_w2v_s": time.time() - t1, "mb_paseos": P.nbytes / 1e6}

def node2vec_activos(A, p, q, dim, **kw):
    act = np.flatnonzero(np.diff(A.indptr) > 0)
    E_, t = node2vec(A[act][:, act], p, q, dim, **kw)
    E = np.zeros((A.shape[0], dim), dtype=np.float32)
    E[act] = E_
    return E, act, t

### SVD como embedder

- Adyacencia normalizada Â = D^(-1/2) · A · D^(-1/2) (la misma que usa el clustering espectral)

- Embedding = U · √Σ de sus `dim` mayores valores singulares: método **lineal**, sin paseos

- **SVD aleatorizada** (Halko et al.): con varias componentes conexas, cada una aporta un valor singular igual a 1 y ARPACK (`svds`) no converge; la aleatorizada no depende de que estén separados

In [ ]:
def embedding_svd(A, dim, seed=SEED):
    d = np.asarray(A.sum(axis=1)).ravel()
    Dm = sp.diags(1 / np.sqrt(np.maximum(d, 1e-12)))
    U, S, _ = randomized_svd((Dm @ A @ Dm).tocsr(), n_components=dim, n_iter=7, random_state=seed)
    return (U * np.sqrt(S)).astype(np.float32)

### Vecindad en el embedding

- `vecinos_emb`: los `k` vecinos más cercanos por **coseno** de cada nodo de la muestra (exacto, por bloques)

- `perfil_vecindad`: qué comparten un nodo y sus vecinos del embedding

    - **Localidad:** % de vecinos que también son vecinos en el grafo

    - **Región, familia, segmento, tipología:** % de vecinos con el mismo atributo (homofilia)

    - **Rol:** Spearman entre el grado del nodo y el grado medio de sus vecinos; alto = parecidos por **posición estructural**

In [ ]:
def normalizar(E):
    return E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)

def vecinos_emb(E, muestra, k, bloque=1000):
    En = normalizar(E)
    out = np.empty((len(muestra), k), dtype=np.int64)
    for b in range(0, len(muestra), bloque):
        m = muestra[b:b + bloque]
        S = En[m] @ En.T
        S[np.arange(len(m)), m] = -np.inf
        top = np.argpartition(-S, k, axis=1)[:, :k]
        out[b:b + bloque] = np.take_along_axis(top, np.argsort(-np.take_along_axis(S, top, 1), 1), 1)
    return out

def _iguales(col, muestra, NN):
    x = pd.Series(col).astype(object).to_numpy()
    a, b = x[muestra][:, None], x[NN]
    valido = pd.notna(a) & pd.notna(b)
    return 100 * (a == b)[valido].mean() if valido.any() else np.nan

def perfil_vecindad(NN, muestra, A, nodos, tipologia):
    k = NN.shape[1]
    local = np.asarray((A > 0)[np.repeat(muestra, k), NN.ravel()]).ravel().mean()
    lg = np.log1p(nodos.grado.to_numpy())
    return {"% vecinos en el grafo": 100 * local,
            "% misma región": _iguales(nodos.region, muestra, NN),
            "% misma familia": _iguales(nodos.familia, muestra, NN),
            "% mismo segmento": _iguales(nodos.segmento, muestra, NN),
            "% misma tipología": _iguales(tipologia, muestra, NN),
            "rol (Spearman grado)": spearmanr(lg[muestra], lg[NN].mean(axis=1))[0]}

def jaccard_vecinos(N1, N2):
    return np.mean([len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(N1, N2)])

### Convergencia entre particiones

- **ARI:** 1 = misma partición, ≈ 0 = como al azar (corrige por azar)

- **NMI:** información compartida entre particiones, de 0 a 1

- **Pureza:** fracción de nodos cuyo grupo latente coincide con la etiqueta mayoritaria del grupo

- Se comparan solo los nodos con etiqueta en ambas; el ruido (−1) de DBSCAN/HDBSCAN queda fuera

In [ ]:
def pureza(lat, ref):
    t = pd.crosstab(lat, ref)
    return t.max(axis=1).sum() / t.to_numpy().sum()

def convergencia(lat, ref):
    m = pd.notna(ref) & (pd.Series(ref).astype(str) != "-1").to_numpy()
    l, r = np.asarray(lat)[m], np.asarray(ref)[m].astype(str)
    return {"n": int(m.sum()), "ari": adjusted_rand_score(r, l), "nmi": normalized_mutual_info_score(r, l),
            "pureza": pureza(l, r)}

### Particiones de las partes anteriores

- `tipologia_proveedor`: nombre de la tipología de KMeans o `None` si no entró al clustering

    - El nombre se asigna por el **perfil** del clúster (más entidades → multicliente; más obras → contratistas; más postor único → sin competencia; resto → típicos), no por su número

    - Spark no garantiza la misma numeración de clústeres entre ejecuciones: mapear por número cambiaría los nombres al re-ejecutar

- `particiones_previas`: para cada proveedor del grafo, su etiqueta de KMeans, DBSCAN, HDBSCAN y Ward (`02_Clustering`)

- `cargar_comunidades`: etiqueta de la Parte V si su artefacto existe; si no, `None`

- `centralidad_previa`: PageRank y HITS de `04_PageRank` por proveedor

In [ ]:
def _leer_opcional(ruta):
    return pd.read_parquet(ruta) if ruta.exists() else None

def particiones_previas(claves):
    out = pd.DataFrame({"proveedor_key": claves})
    for f, col in (("kmeans_proveedor", "cluster"), ("densidad_proveedor", None), ("jerarquico_proveedor", None)):
        d = _leer_opcional(clus_dir / "datos" / f)
        if d is None:
            print(f"Aviso: falta 02_Clustering/temp/datos/{f}")
            continue
        cols = [c for c in d.columns if c != "proveedor_key" and (c == col or c in ("dbscan", "hdbscan", "jerarquico", "ward"))]
        out = out.merge(d[["proveedor_key"] + cols], on="proveedor_key", how="left")
    return out.rename(columns={"cluster": "kmeans"})

def tipologia_proveedor(claves):
    km = _leer_opcional(clus_dir / "datos" / "kmeans_proveedor")
    if km is None:
        print("Aviso: faltan las etiquetas KMeans de 02_Clustering")
        return np.full(len(claves), None)
    f = pd.read_parquet(artef_dir / "features_proveedor", columns=["proveedor_key", "n_entidades", "pct_obras", "pct_postor_unico"])
    perfil = km.merge(f, on="proveedor_key").groupby("cluster")[["n_entidades", "pct_obras", "pct_postor_unico"]].mean()
    nombres = {perfil.n_entidades.idxmax(): "grandes multicliente"}
    nombres[perfil.drop(list(nombres)).pct_obras.idxmax()] = "contratistas de obras"
    nombres[perfil.drop(list(nombres)).pct_postor_unico.idxmax()] = "ganadores sin competencia"
    nombres.update({c: "típicos de pocos clientes" for c in perfil.index if c not in nombres})
    d = pd.DataFrame({"proveedor_key": claves}).merge(km, on="proveedor_key", how="left")
    return d.cluster.map(nombres).to_numpy()

def cargar_comunidades(claves):
    ruta = COMUNIDADES["ruta"]
    if not (ruta / "particion.parquet").exists():
        print(f"Parte V pendiente: no existe {(ruta / 'particion.parquet').relative_to(proyecto)}")
        return None
    principal = json.loads((ruta / "config.json").read_text())["principal"]
    c = (pd.read_parquet(ruta / "nodos.parquet", columns=["idx", COMUNIDADES["llave"]])
           .merge(pd.read_parquet(ruta / "particion.parquet", columns=["idx", principal]), on="idx"))
    c.columns = ["idx", "proveedor_key", "comunidad"]
    print(f"Parte V disponible ({principal}): {c.comunidad.nunique():,} comunidades en {len(c):,} proveedores del núcleo")
    return pd.DataFrame({"proveedor_key": claves}).merge(c, on="proveedor_key", how="left").comunidad.to_numpy()

def centralidad_previa(claves):
    nodos, pr, hits = (_leer_opcional(pr_dir / f) for f in ("nodos.parquet", "pagerank.parquet", "hits.parquet"))
    if nodos is None or pr is None:
        print("Aviso: faltan las salidas de 04_PageRank; se sigue sin PageRank/HITS")
        return None
    col_pr = "pr" if "pr" in pr.columns else [c for c in pr.columns if c.startswith("pr")][0]
    n = nodos[nodos.tipo == "proveedor"][["idx", "clave"]].merge(pr[["idx", col_pr]], on="idx")
    if hits is not None:
        n = n.merge(hits, on="idx", how="left")
    n = n.rename(columns={"clave": "proveedor_key", col_pr: "pagerank"}).drop(columns="idx")
    return pd.DataFrame({"proveedor_key": claves}).merge(n, on="proveedor_key", how="left")

### Predicción de enlaces

- `grafo_anios`: co-participación sumando solo los años dados (el grafo que "se conoce" al entrenar)

- `pares_nuevos`: aristas del año objetivo entre nodos activos que **no** estaban en el grafo previo (positivos)

- `negativos`: pares sin arista en el grafo previo ni en el año objetivo; una fracción al azar y otra a **2 saltos** (difíciles: comparten un vecino)

- **Operadores de par** (Grover & Leskovec): Hadamard u ⊙ v, L1 |u − v| y L2 (u − v)²; son simétricos, el orden del par no importa

- **Heurísticas de referencia:** vecinos comunes, Jaccard y Adamic-Adar sobre el grafo de entrenamiento

In [ ]:
def grafo_anios(aristas_anio, anios_, n):
    d = aristas_anio[aristas_anio.anio.isin(anios_)].groupby(["i", "j"], as_index=False).peso.sum()
    return matriz_adyacencia(d.i.to_numpy(), d.j.to_numpy(), d.peso.to_numpy(), n)

def pares_nuevos(aristas_anio, anio_obj, A_prev, activos):
    d = aristas_anio[(aristas_anio.anio == anio_obj) & aristas_anio.i.isin(activos) & aristas_anio.j.isin(activos)]
    previo = np.asarray(A_prev[d.i.to_numpy(), d.j.to_numpy()]).ravel() > 0
    return d.loc[~previo, ["i", "j"]].drop_duplicates().to_numpy()

def _llave(i, j, n):
    return np.minimum(i, j).astype(np.int64) * n + np.maximum(i, j)

def _vecino_uniforme(A, nodos, rng):
    ini, fin = A.indptr[nodos], A.indptr[nodos + 1]
    return A.indices[ini + (rng.random(len(nodos)) * (fin - ini)).astype(np.int64)]

def negativos(A_prev, excluir, activos, cuantos, frac_dificil, rng):
    n = A_prev.shape[0]
    excl = set(excluir.tolist())
    out, vistos = [], set()
    for dificil, meta in ((True, int(cuantos * frac_dificil)), (False, cuantos - int(cuantos * frac_dificil))):
        tomados = 0
        while tomados < meta:
            u = rng.choice(activos, 2 * meta)
            x = _vecino_uniforme(A_prev, _vecino_uniforme(A_prev, u, rng), rng) if dificil else rng.choice(activos, 2 * meta)
            ok = (u != x) & (np.asarray(A_prev[u, x]).ravel() == 0)
            for a_, b_ in zip(u[ok], x[ok]):
                k = int(_llave(a_, b_, n))
                if k in excl or k in vistos:
                    continue
                vistos.add(k); out.append((min(a_, b_), max(a_, b_), int(dificil))); tomados += 1
                if tomados == meta:
                    break
    return np.array(out)

OPERADORES = {"hadamard": lambda u, v: u * v, "l1": lambda u, v: np.abs(u - v), "l2": lambda u, v: (u - v) ** 2}

def features_par(E, i, j, op):
    return OPERADORES[op](E[i], E[j])

def heuristicas(A, i, j):
    B = (A > 0).astype(np.float32).tocsr()
    grado = np.asarray(B.sum(axis=1)).ravel()
    inv_log = sp.diags(1 / np.log(np.maximum(grado, 2)))
    cn = np.asarray(B[i].multiply(B[j]).sum(axis=1)).ravel()
    aa = np.asarray((B[i] @ inv_log).multiply(B[j]).sum(axis=1)).ravel()
    jac = cn / np.maximum(grado[i] + grado[j] - cn, 1)
    return {"vecinos_comunes": cn, "jaccard": jac, "adamic_adar": aa}

def metricas_ranking(y, s):
    return {"roc_auc": roc_auc_score(y, s), "pr_auc": average_precision_score(y, s)}